# Notebook 1: Data Exploration

Explore the NTNU Autoferry Sensor Fusion Dataset.

**Sensors:**
- Lidar (ID=1): Active, 2D position (N, E)
- Radar (ID=2): Active, 2D position (N, E)
- IR Camera (ID=3): Passive, bearing only
- EO Camera (ID=4): Passive, bearing only

In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from helpers import load_scenario, get_all_detections, get_ground_truth, get_ownship

%matplotlib inline
plt.rcParams['figure.figsize'] = (14, 8)

## 1.1 Load Scenario Data

In [ ]:
SCENARIO = 'scenario2'
loader = load_scenario(SCENARIO)
detections = get_all_detections(loader)
ground_truth = get_ground_truth(loader)
ownship = get_ownship(loader)

print('Loaded', len(detections), 'sensors')
print('Ground truth targets:', list(ground_truth.keys()))
print('Ownship trajectory shape:', len(ownship))

## 1.2 Sensor Detection Overview

In [ ]:
for sensor_id, df in detections.items():
    print('
=== Sensor', sensor_id, '===')
    print('Shape:', df.shape)
    print('Columns:', list(df.columns))
    print('Time range:', df['time'].min(), '-', df['time'].max(), 's')
    print(df.head(3))

## 1.3 Plot Scenario Overview

In [ ]:
fig, ax = plt.subplots(figsize=(12, 8))

for sensor_id, df in detections.items():
    ax.scatter(df['x_piren'], df['y_piren'], s=2, alpha=0.4, label='Sensor ' + str(sensor_id))

for target_id, gt_df in ground_truth.items():
    ax.plot(gt_df['x_piren'], gt_df['y_piren'], 'k-', linewidth=2, label='Target ' + str(target_id))

ax.plot(ownship['x_piren'], ownship['y_piren'], 'r--', linewidth=1, label='Ownship')
ax.set_xlabel('East (m)')
ax.set_ylabel('North (m)')
ax.set_title(SCENARIO + ' Overview')
ax.legend()
ax.grid(True)
ax.set_aspect('equal')
plt.tight_layout()
plt.show()

## 1.4 Detection Timeline

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

for sensor_id, df in detections.items():
    counts = df.groupby(df['time'].round()).size()
    ax.plot(counts.index, counts.values, label='Sensor ' + str(sensor_id), linewidth=2)

ax.set_xlabel('Time (s)')
ax.set_ylabel('Detection Count')
ax.set_title(SCENARIO + ' Detection Timeline')
ax.legend()
ax.grid(True)
plt.tight_layout()
plt.show()

## 1.5 Individual Sensor Plots

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 12))
axes = axes.flatten()

for idx, (sensor_id, df) in enumerate(detections.items()):
    ax = axes[idx]
    ax.scatter(df['x_piren'], df['y_piren'], s=3, alpha=0.5)
    
    for tid, gt in ground_truth.items():
        ax.plot(gt['x_piren'], gt['y_piren'], 'k-', linewidth=1.5)
    
    ax.set_title('Sensor ' + str(sensor_id))
    ax.set_xlabel('East (m)')
    ax.set_ylabel('North (m)')
    ax.grid(True)
    ax.set_aspect('equal')

plt.suptitle(SCENARIO + ' All Sensors', fontsize=14)
plt.tight_layout()
plt.show()

## 1.6 Ground Truth Trajectories

In [ ]:
fig, ax = plt.subplots(figsize=(12, 8))

for target_id, gt_df in ground_truth.items():
    ax.plot(gt_df['x_piren'], gt_df['y_piren'], linewidth=2, label='Target ' + str(target_id))

ax.plot(ownship['x_piren'], ownship['y_piren'], 'r--', linewidth=1, label='Ownship')
ax.set_xlabel('East (m)')
ax.set_ylabel('North (m)')
ax.set_title(SCENARIO + ' Ground Truth Trajectories')
ax.legend()
ax.grid(True)
ax.set_aspect('equal')
plt.tight_layout()
plt.show()

## 1.7 Sensor Statistics

In [ ]:
from helpers import compute_sensor_metrics

for sensor_id, df in detections.items():
    metrics = compute_sensor_metrics(df, ground_truth, sensor_id)
    print('
=== Sensor', sensor_id, '===')
    for k, v in metrics.items():
        print(k + ':', round(v, 4) if isinstance(v, float) else v)